In [ ]:
#!/usr/bin/env python3
import os
from itertools import product
from textwrap import dedent

# --------- Adjust these if needed ----------
PYTHONPATH_DIR = "/home/hpc/iwi5/iwi5295h/PIGNN-Attn-LS"
DATA_PARQUET   = "/home/woody/iwi5/iwi5295h/large/HVN_50000_NR_plain_4_to_512_buses.parquet"
PARTITION      = "a100"          # SLURM partition
GPU_GRES       = "gpu:a100:1"    # --gres
GPU_CONSTRAINT = "a100_80"       # -C constraint
TIME_LIMIT     = "24:00:00"
CPUS_PER_TASK  = 16
EPOCHS         = 100
NUM_ATTN_LAYERS = 1              # 'attn1' in file name

# Output directories (scripts and slurm logs)
SCRIPT_OUT_DIR = "."             # where .sh files will be written
JOB_OUT_DIR    = "./Job_out"     # used in #SBATCH --output/--error lines

# Hyperparameter grids
d_hi_list  = [128, 256]          # Capacity
n_heads_list = [4, 8]
d_list     = [16, 32]
K_list     = [40, 80]
# -------------------------------------------

os.makedirs(JOB_OUT_DIR, exist_ok=True)
os.makedirs(SCRIPT_OUT_DIR, exist_ok=True)

def make_script(name: str, d: int, d_hi: int, n_heads: int, K: int) -> str:
    out_log = os.path.join(JOB_OUT_DIR, f"{name}.out")
    err_log = os.path.join(JOB_OUT_DIR, f"{name}.err")
    # Build the .sh content
    return dedent(f"""\
    #!/bin/bash

    #SBATCH --job-name={name}
    #SBATCH --output={out_log}
    #SBATCH --error={err_log}
    #SBATCH --gres={GPU_GRES} -C {GPU_CONSTRAINT}      # Request 1 A100 GPU
    #SBATCH --time={TIME_LIMIT}                        # Max runtime
    #SBATCH --ntasks=1                                 # Single task
    #SBATCH --cpus-per-task={CPUS_PER_TASK}            # CPU cores
    #SBATCH --partition={PARTITION}                    # Partition

    export PYTHONPATH={PYTHONPATH_DIR}:$PYTHONPATH
    module load python

    cd ..

    # Run your application or script
    srun python train_valid_test.py \\
        --d={d} \\
        --d_hi={d_hi} \\
        --n_heads={n_heads} \\
        --num_attn_layers={NUM_ATTN_LAYERS} \\
        --K={K} \\
        --EPOCHS={EPOCHS} \\
        --lr_scheduler=CosineAnnealingLR \\
        --PARQUET {DATA_PARQUET} \\
        --vlimit \\
        --use_armijo \\
        --model GNSMsg_EdgeSelfAttn
    """)

count = 0
for d_hi, n_heads, d, K in product(d_hi_list, n_heads_list, d_list, K_list):
    job_name = f"attn{NUM_ATTN_LAYERS}_d{d}_dhi{d_hi}_h{n_heads}_k{K}_HVN_50000_NR_plain_4_to_512"
    script_path = os.path.join(SCRIPT_OUT_DIR, f"{job_name}.sh")
    with open(script_path, "w", encoding="utf-8~~~~") as f:
        f.write(make_script(job_name, d=d, d_hi=d_hi, n_heads=n_heads, K=K))
    os.chmod(script_path, 0o755)
    count += 1

print(f"Generated {count} scripts in: {os.path.abspath(SCRIPT_OUT_DIR)}")
print(f"Logs will go to: {os.path.abspath(JOB_OUT_DIR)}")

In [ ]:
for f in *.sh; do sbatch "$f"; done

In [ ]:
attn1_d16_dhi128_h8_k80_HVN_50000_NR_plain_4_to_512.sh attn1_d16_dhi256_h4_k80_HVN_50000_NR_plain_4_to_512.sh attn1_d16_dhi256_h8_k80_HVN_50000_NR_plain_4_to_512.sh attn1_d32_dhi128_h4_k40_HVN_50000_NR_plain_4_to_512.sh attn1_d32_dhi128_h4_k80_HVN_50000_NR_plain_4_to_512.sh attn1_d32_dhi128_h8_k40_HVN_50000_NR_plain_4_to_512.sh attn1_d32_dhi128_h8_k80_HVN_50000_NR_plain_4_to_512.sh attn1_d32_dhi256_h4_k40_HVN_50000_NR_plain_4_to_512.sh attn1_d32_dhi256_h4_k80_HVN_50000_NR_plain_4_to_512.sh attn1_d32_dhi256_h8_k40_HVN_50000_NR_plain_4_to_512.sh attn1_d32_dhi256_h8_k80_HVN_50000_NR_plain_4_to_512.sh

attn1_d16_dhi128_h4_k40_HVN_50000_NR_plain_4_to_512.sh
attn1_d16_dhi128_h4_k80_HVN_50000_NR_plain_4_to_512.sh
attn1_d16_dhi128_h8_k40_HVN_50000_NR_plain_4_to_512.sh
attn1_d16_dhi256_h4_k40_HVN_50000_NR_plain_4_to_512.sh
attn1_d16_dhi256_h8_k40_HVN_50000_NR_plain_4_to_512.sh